# 03 — ORESTAR City Council fundraising profiles (2024)

Candidate fundraising profiles for **Portland City Council** using the cleaned
ORESTAR source for 2024.

The profile representation is kept identical across years:

- total contribution amount and contribution-record count;
- mean, median, minimum, maximum, and standard deviation;
- cash vs. in-kind contributions;
- Micro / Small / Medium / Large / Mega bins;
- amount and count shares by bin.

Confirmed candidate linkage is attached when available. Source-only candidates
are retained rather than silently dropped.


## 1. Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 190)

cwd = Path.cwd().resolve()

ROOT = cwd
while not (ROOT / "pyproject.toml").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError(
            "Could not find repository root. "
            "Expected a pyproject.toml file somewhere above this notebook."
        )
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from helpers.paths import (
    CLEAN,
    PROCESSED,
    fundraising_processed_dir,
    orestar_file_audit_path,
    orestar_transactions_path,
    spending_processed_dir,
)

print("ROOT:", ROOT)

YEAR = 2024
CONTEST = "city_council"

TRANSACTIONS_PATH = orestar_transactions_path(
    YEAR,
    CONTEST,
)

AUDIT_PATH = orestar_file_audit_path(
    YEAR,
    CONTEST,
)

CROSSWALK_PATH = (
    PROCESSED
    / "candidates"
    / f"candidate_source_crosswalk_{YEAR}.csv"
)

OUTPUT_DIR = fundraising_processed_dir(
    YEAR,
    CONTEST,
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Transactions:", TRANSACTIONS_PATH)
print("Audit:", AUDIT_PATH)
print("Crosswalk exists:", CROSSWALK_PATH.exists())
print("Output:", OUTPUT_DIR)


## 2. Load and validate cleaned City Council ORESTAR

In [ ]:
def to_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )

transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    low_memory=False,
)

required = {
    "year",
    "contest_type",
    "office",
    "district",
    "source_file_stem",
    "sub_type",
    "is_reported_contribution",
    "reported_contribution_amount",
}

missing = sorted(
    required
    - set(transactions.columns)
)

if missing:
    raise ValueError(
        f"Missing clean ORESTAR columns: {missing}"
    )

contest_values = set(
    transactions["contest_type"]
    .dropna()
    .astype(str)
    .unique()
)

if contest_values != {CONTEST}:
    raise ValueError(
        f"Expected only {CONTEST}; found {sorted(contest_values)}"
    )

print("Rows:", f"{len(transactions):,}")
print("Districts:", sorted(
    transactions["district"]
    .dropna()
    .unique()
    .tolist()
))

display(
    transactions["sub_type"]
    .value_counts(dropna=False)
    .to_frame("rows")
)


## 3. Keep positive reported contributions

The cleaner already defines `Cash Contribution` and `In-Kind Contribution` as
positive reported contributions and excludes refunds.


In [ ]:
transactions["reported_contribution_amount"] = pd.to_numeric(
    transactions["reported_contribution_amount"],
    errors="coerce",
)

contributions = transactions.loc[
    to_bool(
        transactions["is_reported_contribution"]
    )
    & transactions["reported_contribution_amount"].notna()
    & transactions["reported_contribution_amount"].gt(0)
].copy()

contributions["amount"] = (
    contributions["reported_contribution_amount"]
)

print("Contribution records:", f"{len(contributions):,}")
print("Total amount:", f"${contributions['amount'].sum():,.2f}")


## 4. File audit

In [ ]:
if AUDIT_PATH.exists():
    audit = pd.read_csv(
        AUDIT_PATH,
        low_memory=False,
    )

    duplicate_exports = audit.loc[
        to_bool(
            audit["is_exact_duplicate_export"]
        )
    ].copy()

    print(
        "Exact duplicate export rows:",
        len(duplicate_exports),
    )

    if len(duplicate_exports):
        display(
            duplicate_exports[
                [
                    "district",
                    "source_file",
                    "source_file_hash",
                    "same_hash_file_count",
                ]
            ]
        )
else:
    print("No file audit found.")


## 5. Candidate identity

When a confirmed ORESTAR crosswalk exists, use the official candidate key.
Otherwise preserve a stable source-level profile key.


In [ ]:
def attach_orestar_identity(data: pd.DataFrame, crosswalk_path: Path) -> pd.DataFrame:
    frame = data.copy()

    frame["source_candidate_name"] = (
        frame["source_file_stem"]
        .astype("string")
        .str.strip()
    )

    frame["canonical_candidate"] = pd.NA
    frame["candidate_key"] = pd.NA
    frame["linkage_status"] = "source_only"

    if crosswalk_path.exists():
        crosswalk = pd.read_csv(
            crosswalk_path,
            low_memory=False,
        )

        needed = {
            "source",
            "classification",
            "year",
            "district",
            "source_candidate_name",
            "suggested_candidate",
            "suggested_candidate_key",
        }

        if needed.issubset(crosswalk.columns):
            matched = (
                crosswalk.loc[
                    crosswalk["source"].eq("orestar")
                    & crosswalk["classification"].eq("match"),
                    [
                        "year",
                        "district",
                        "source_candidate_name",
                        "suggested_candidate",
                        "suggested_candidate_key",
                    ],
                ]
                .rename(
                    columns={
                        "suggested_candidate": "_canonical_candidate",
                        "suggested_candidate_key": "_candidate_key",
                    }
                )
                .drop_duplicates()
            )

            frame = frame.merge(
                matched,
                on=[
                    "year",
                    "district",
                    "source_candidate_name",
                ],
                how="left",
                validate="many_to_one",
            )

            frame["canonical_candidate"] = frame["_canonical_candidate"]
            frame["candidate_key"] = frame["_candidate_key"]

            frame["linkage_status"] = np.where(
                frame["candidate_key"].notna(),
                "matched_to_official_candidate",
                "unmatched_source_candidate",
            )

            frame = frame.drop(
                columns=[
                    "_canonical_candidate",
                    "_candidate_key",
                ]
            )

    frame["candidate"] = (
        frame["canonical_candidate"]
        .fillna(frame["source_candidate_name"])
    )

    source_key = (
        frame["year"].astype("Int64").astype(str)
        + "|"
        + frame["district"].astype("Int64").astype(str)
        + "|orestar|"
        + frame["source_candidate_name"]
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    frame["profile_key"] = (
        frame["candidate_key"]
        .fillna(source_key)
    )

    return frame

contributions = attach_orestar_identity(
    contributions,
    CROSSWALK_PATH,
)

identity_summary = (
    contributions[
        [
            "year",
            "district",
            "source_file_stem",
            "candidate",
            "candidate_key",
            "profile_key",
            "linkage_status",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "district",
            "candidate",
        ]
    )
)

display(identity_summary)

print(
    "\nLinkage status:"
)
display(
    identity_summary["linkage_status"]
    .value_counts(dropna=False)
    .to_frame("candidate_source_rows")
)


## 6. Candidate fundraising summary

In [ ]:
PROFILE_KEYS = [
    "year",
    "contest_type",
    "office",
    "district",
    "profile_key",
    "candidate",
]

candidate_summary = (
    contributions
    .groupby(
        PROFILE_KEYS,
        as_index=False,
        dropna=False,
    )
    .agg(
        total_amount=("amount", "sum"),
        total_contribution_count=("amount", "size"),
        average_contribution=("amount", "mean"),
        median_contribution=("amount", "median"),
        min_contribution=("amount", "min"),
        max_contribution=("amount", "max"),
        std_contribution=("amount", "std"),
    )
)

type_summary = (
    contributions
    .pivot_table(
        index=PROFILE_KEYS,
        columns="sub_type",
        values="amount",
        aggfunc=[
            "sum",
            "size",
        ],
        fill_value=0,
    )
)

type_summary.columns = [
    "_".join(
        str(x)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
        for x in col
    )
    for col in type_summary.columns
]

type_summary = (
    type_summary
    .reset_index()
)

candidate_summary = candidate_summary.merge(
    type_summary,
    on=PROFILE_KEYS,
    how="left",
    validate="one_to_one",
)

identity_cols = (
    contributions[
        [
            "profile_key",
            "source_file_stem",
            "source_candidate_name",
            "canonical_candidate",
            "candidate_key",
            "linkage_status",
        ]
    ]
    .drop_duplicates(
        subset=["profile_key"]
    )
)

candidate_summary = candidate_summary.merge(
    identity_cols,
    on="profile_key",
    how="left",
    validate="one_to_one",
)

candidate_summary = candidate_summary.sort_values(
    [
        "district",
        "total_amount",
    ],
    ascending=[
        True,
        False,
    ],
)

display(candidate_summary)


## 7. Contribution-size bins

In [ ]:
BIN_LABELS = [
    "Micro",
    "Small",
    "Medium",
    "Large",
    "Mega",
]

BIN_EDGES = [
    -np.inf,
    25,
    100,
    250,
    1000,
    np.inf,
]

contributions["contribution_bin"] = pd.cut(
    contributions["amount"],
    bins=BIN_EDGES,
    labels=BIN_LABELS,
    right=True,
    ordered=True,
)

display(
    contributions["contribution_bin"]
    .value_counts(sort=False)
    .to_frame("records")
)


## 8. Long fundraising profile

In [ ]:
profile_long = (
    contributions
    .groupby(
        PROFILE_KEYS
        + [
            "contribution_bin",
        ],
        observed=False,
        as_index=False,
        dropna=False,
    )
    .agg(
        amount=("amount", "sum"),
        contribution_count=("amount", "size"),
    )
)

profile_totals = (
    profile_long
    .groupby(
        PROFILE_KEYS,
        as_index=False,
        dropna=False,
    )
    .agg(
        profile_total_amount=("amount", "sum"),
        profile_total_contribution_count=(
            "contribution_count",
            "sum",
        ),
    )
)

profile_long = profile_long.merge(
    profile_totals,
    on=PROFILE_KEYS,
    how="left",
    validate="many_to_one",
)

profile_long["amount_share"] = (
    profile_long["amount"]
    / profile_long["profile_total_amount"]
)

profile_long["contribution_share"] = (
    profile_long["contribution_count"]
    / profile_long["profile_total_contribution_count"]
)

profile_long = profile_long.merge(
    identity_cols,
    on="profile_key",
    how="left",
    validate="many_to_one",
)

display(profile_long.head(15))


## 9. Validate profile shares

In [ ]:
validation = (
    profile_long
    .groupby(
        PROFILE_KEYS,
        as_index=False,
        dropna=False,
    )
    .agg(
        amount_share_sum=("amount_share", "sum"),
        contribution_share_sum=("contribution_share", "sum"),
    )
)

validation["amount_share_ok"] = np.isclose(
    validation["amount_share_sum"],
    1.0,
)

validation["contribution_share_ok"] = np.isclose(
    validation["contribution_share_sum"],
    1.0,
)

print(
    "All amount-share profiles valid:",
    validation["amount_share_ok"].all(),
)

print(
    "All contribution-share profiles valid:",
    validation["contribution_share_ok"].all(),
)

display(
    validation.loc[
        ~validation["amount_share_ok"]
        | ~validation["contribution_share_ok"]
    ]
)


## 10. Wide fundraising profile

In [ ]:
metrics = [
    "amount",
    "amount_share",
    "contribution_count",
    "contribution_share",
]

wide_parts = []

for metric in metrics:
    part = (
        profile_long
        .pivot(
            index=PROFILE_KEYS,
            columns="contribution_bin",
            values=metric,
        )
        .reindex(columns=BIN_LABELS)
        .fillna(0)
    )

    part.columns = [
        f"{metric}_{str(bin_name).lower()}"
        for bin_name in part.columns
    ]

    wide_parts.append(part)

profile_wide = pd.concat(
    wide_parts,
    axis=1,
).reset_index()

profile_wide = profile_wide.merge(
    candidate_summary,
    on=PROFILE_KEYS,
    how="left",
    validate="one_to_one",
)

print("Rows:", len(profile_wide))
print("Columns:", len(profile_wide.columns))
display(profile_wide.head())


## 11. Quick descriptive view

In [ ]:
district_summary = (
    candidate_summary
    .groupby(
        [
            "year",
            "district",
        ],
        as_index=False,
    )
    .agg(
        candidate_source_rows=("profile_key", "nunique"),
        total_fundraising=("total_amount", "sum"),
        median_candidate_fundraising=("total_amount", "median"),
        min_candidate_fundraising=("total_amount", "min"),
        max_candidate_fundraising=("total_amount", "max"),
    )
)

display(district_summary)


## 12. Export

In [ ]:
summary_path = (
    OUTPUT_DIR
    / "orestar_candidate_fundraising_summary.csv"
)

long_path = (
    OUTPUT_DIR
    / "orestar_candidate_fundraising_profiles_long.csv"
)

wide_path = (
    OUTPUT_DIR
    / "orestar_candidate_fundraising_profiles_wide.csv"
)

candidate_summary.to_csv(
    summary_path,
    index=False,
)

profile_long.to_csv(
    long_path,
    index=False,
)

profile_wide.to_csv(
    wide_path,
    index=False,
)

print("SAVED", summary_path)
print("SAVED", long_path)
print("SAVED", wide_path)
